In [0]:
import requests
import json
import os
from datetime import datetime

In [0]:
# Define o caminho base para salvar os arquivos de partidas
BASE_PATH = "/Volumes/workspace/project_data_football_raw/partidas_raw"

# Cria o diretório se não existir
os.makedirs(BASE_PATH, exist_ok=True)

In [0]:
def run():
    # Inicia o processo de ingestão dinâmica de partidas
    print("Iniciando ingestão dinâmica de partidas...")
    
    rodada = 1
    continuar = True
    
    while continuar:
        # Monta a URL para buscar os dados da rodada atual
        url = f"https://api.cartola.globo.com/partidas/{rodada}"
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            
            if data.get("partidas"):
                # Cria o caminho da pasta da rodada
                path_rodada = f"{BASE_PATH}/rodada={rodada}"
                dbutils.fs.mkdirs(path_rodada)
                
                # Define o nome do arquivo dentro da pasta da rodada com timestamp
                file_name = f"{path_rodada}/data_rodada{rodada}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
                
                # Salva os dados da rodada em arquivo JSON
                dbutils.fs.put(file_name, json.dumps(data), overwrite=True)
                print(f"Sucesso: Rodada {rodada} processada e salva em subpasta.")
                rodada += 1
            else:
                # Encerra o loop se não houver dados de partidas
                print(f"Finalizado: Rodada {rodada} não possui dados.")
                continuar = False
        else:
            # Encerra o loop em caso de erro na requisição
            continuar = False

    # Finaliza o pipeline de ingestão
    print("Pipeline de ingestão concluído!")
run()